# tRNA sequencing - data processing example

This notebook is an example of processing raw data from tRNA sequencing.
It starts from raw paired-end reads and concludes with a datafile containing counts of tRNA transcripts, with information about amino acids, codons, charge etc. and broken down based on sample name, replicate etc.
Some generically useful plots will be generated, such as: logo plots of the UMI sequences and non-template bases, plots showing the transcript/codon coverage and charge per sample.

## Setup - no need to change
Setting up folders, alignment setting, sequence information etc.
There should be no need to change these setting if the raw bz2 compressed data is stored in a folder named `data/raw_fastq`.

### Importing external packages

In [1]:
# These magic cells reloads imported code when it changes.
# Very useful for prototyping, other will do nothing.
%reload_ext autoreload
%autoreload 2

### General imports ###
import os, sys, shutil, bz2
from pathlib import Path
import pandas as pd
pd.set_option('display.max_columns', 50)
import numpy as np

### Plotting imports ###
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import matplotlib.colors as mcolors
import matplotlib as mpl
from matplotlib.patches import StepPatch
import matplotlib.ticker as ticker
import matplotlib.gridspec as gridspec
import logomaker as lm
from cycler import cycler
palette = list(mcolors.TABLEAU_COLORS.keys())
sns.set_theme(style="ticks", palette="muted")
sns.set_context("talk")

%matplotlib inline

### Custom functions


In [2]:
## Log the date
from datetime import datetime
logDate = str(datetime.now().date())
print ('Current date: {}'.format(logDate))

## Function to log the status
def log_status(message):
    """Log the given message with a timestamp to a file."""
    with open("CurrentStatus.log", "a") as log_file:
        # Write the current timestamp and the message to the log
        log_file.write(f"{datetime.now()}: {message}\n")

## Function to find the repo directory
def find_tooldir(start_dir, target_dir_name):
    """
    Look for a specific directory (target_dir_name) starting from start_dir and expanding the search
    to parent, sibling, and cousin directories.
    """
    visited = set()  # Keep track of visited directories to avoid loops

    def search_dir(current_dir):
        if current_dir in visited or not os.path.isdir(current_dir):
            return None
        visited.add(current_dir)

        # Check current directory
        target_path = os.path.join(current_dir, target_dir_name)
        if os.path.isdir(target_path):
            return target_path

        parent_dir = os.path.dirname(current_dir)
        siblings_and_cousins = os.listdir(parent_dir) if os.path.isdir(parent_dir) else []

        for relative in siblings_and_cousins:
            relative_path = os.path.join(parent_dir, relative)
            if os.path.isdir(relative_path) and relative != os.path.basename(current_dir):
                # Check inside sibling/cousin directories
                found = search_dir(relative_path)
                if found:
                    return found

        # Recurse up to parent directory if target not found in current or any sibling/cousin directory
        return search_dir(parent_dir)

    return search_dir(start_dir)

## Function to rename MiSeq files
def rename_MiSeq_file(original_filename):
    parts = original_filename.split('_')
    sample = parts[0]  # Get the sample name which is the first part
    read_indicator = parts[-2]  # The second last element is the read indicator
    new_filename = f"{sample}_{read_indicator}.fastq.bz2"
    return new_filename

## Function to rename NovaSeq files
def rename_NovaSeq_file(original_filename):
    parts = original_filename.split('_')
    sample = parts[2]  # Get the sample name which is the third part
    read_indicator = parts[-2]  # The second last element is the read indicator
    new_filename = f"{sample}_{read_indicator}.fastq.bz2"
    return new_filename


Current date: 2026-02-11


### Setting home directory
Navigate back to `NBdir` when re-running this code block. <br><br>Hint: Make sure the path is **'path/to/experiment'** not **'path/to/experiment/data'**

In [3]:
if not 'NBdir' in globals():
    NBdir = os.getcwd()
print('Notebook is in: {}'.format(NBdir))
os.chdir(NBdir)  # If you changed the current working dir, this will take you back to the notebook dir.

Notebook is in: /mnt/g/GitHub-russel-vincent25/tRNA-charge-seq/projects/example


Define the path to the repo folder and insert it to the system path to enable code import from the `src/` folder.
__If the repo structure is changed this may need to be edited.__

In [4]:
# homedir = '/'.join(NBdir.split('/')[0:-2]) 
# assert(os.path.isdir(homedir))
# print('Repo is in: {}'.format(homedir))
# sys.path.insert(1, homedir)

Alternatively, you can programatically define the tools_dir to look for and the get_toolsdir will look for the correct tools directory in parent and sibling folders. <br><br>Hint: Make sure the path is 'path/to/BioinformaticTools/tRNA-charge-seq'

In [5]:
# Here, 'tooldir' should be replaced with the directory name defined by the user
tooldir = "tRNA-charge-seq"  # User should replace this with the desired directory name

# Find the repository directory by looking for 'tooldir'
homedir = find_tooldir(NBdir, tooldir)

if homedir:
    print(f'Tools directory = {homedir}')
    sys.path.insert(1, homedir)
else:
    print("Couldn't find the specified directory.")

Tools directory = /mnt/g/GitHub-russel-vincent25/tRNA-charge-seq


### Importing internal code
Import all the relevant classes and functions from the `src/` folder:

In [6]:
from src.misc import index_to_sample_df, downsample_raw_input, read_tRNAdb_info, sample_df_to_dict, show_PDF
from src.read_processing import AR_merge, BC_split, Kmer_analysis, BC_analysis, UMI_trim, _5p_trim
from src.alignment import SWIPE_align
from src.stats_collection import STATS_collection
from src.plotting import TRNA_plot
from src.transcript_mutations import TM_analysis

### Specify input 
Default folder names for data and raw fastq files relative to the folder in which this notebook is in.
In this example all the output data should be put in the `data_dir` called `data` and the raw fastq files are stored in the `raw_fastq` folder, under the `data` folder.

In [7]:
data_dir = 'data'
seq_dir = 'raw_fastq'
seq_dir_noDS = seq_dir # Not downsampled
assert(os.path.isdir('{}/{}/{}'.format(NBdir,data_dir,seq_dir)))

In [8]:
# List the files present in the data folder
directory = '{}/{}/{}'.format(NBdir,data_dir,seq_dir)
fileList = os.listdir(directory)

print('Files in the folder:')
for i, filename in enumerate(fileList):
    # Check if filename matches the pattern to avoid renaming unrelated files
    if "_S" in filename and "_L00" in filename and filename.endswith(".fastq.bz2"):
        #new_filename = rename_MiSeq_file(filename)
        new_filename = rename_NovaSeq_file(filename)
        os.rename(os.path.join(directory, filename), os.path.join(directory, new_filename))
        print('Renamed {} to {}'.format(filename, new_filename))
    else:
        print(f'{i}: {filename}')

Files in the folder:
0: P3_R1.fastq.bz2
1: P3_R2.fastq.bz2


The tRNA database is specied as a dictionary with keys for the different species:

In [9]:
tRNA_database = dict()
# Add as many tRNA databases as you want to use if the tRNA reference dataset is updated in the BioinformaticTools/tRNA-charge-seq/tRNA_database folder
tRNA_database['human'] = '{}/tRNA_database/human/hg38-tRNAs.fa'.format(homedir)
tRNA_database['ecoli'] = '{}/tRNA_database/ecoli/ecoli-tRNAs.fa'.format(homedir)

# Modified tRNA library databases - often has the synthetic tRNA constructs
tRNA_database['YLDC'] = '{}/tRNA_database/YLDC/YLDC_hg38-tRNAs.fa'.format(homedir)
tRNA_database['ecoli_EKRV'] = '{}/tRNA_database/ecoli_EKRV/ecoli-tRNAs_EKRV.fa'.format(homedir)

assert(all(os.path.isfile(db) for sp, db in tRNA_database.items()))

tRNA sequencing yields many duplicated reads.
Adding these commonly seen sequences to a list prevents running the same alignments again and again.
This is optional.

In [10]:
common_seqs = '{}/utils/common-seqs.fasta.bz2'.format(homedir)
# ^^ That one is big an comprehensive, good for a full scale run
# with >1e6 reads per sample. For this example though it is going to
# be a bottleneck for alignment so instead take the top 1k:
#common_seqs = '{}/utils/common-seqs_1k.fasta.bz2'.format(homedir)
assert(os.path.isfile(common_seqs))

Scoring matrices for SWIPE alignemnt.
Matrix one is a standard +1 for match and -2 for mismatch.
Matrix two is similar, but also rewarding match to a "masked" nt. marked by the character "N".

In [11]:
SWIPE_score_mat = '{}/utils/nuc_score-matrix.txt'.format(homedir)
SWIPE_score_mat2 = '{}/utils/nuc_score-matrix_2.txt'.format(homedir) # For masked reference sequences
assert(os.path.isfile(SWIPE_score_mat))
assert(os.path.isfile(SWIPE_score_mat2))

Excel file to specify the sequence barcodes/indices e.g. the sequence of Illumina D501 index and the barcode sequence of each adapter.

In [12]:
# The index_list contains 
index_list_path = '{}/utils/index_list_updated.xlsx'.format(homedir)
assert(os.path.isfile(index_list_path))

### Specify output directories
These folder names are used in subsequent processing steps to dump data into the `data_dir`.
Best to not change if you want to reload the data from past processing.

In [13]:
AdapterRemoval_dir = 'AdapterRemoval'
BC_dir = 'BC_split'
UMI_dir = 'UMI_trimmed'
align_dir = 'SWalign'
stats_dir = 'stats_collection'
charge_dir = 'charge_analysis'
TM_dir = 'transcript_mutations'
plotting_dir = 'plotting'

### Specify setting for read length and alignment
Define minimum read length based on minimum insert size.

In [14]:
MIN_INSERT_LEN = 10
UMI_LEN = 11
BC_MAX_LEN = 10
MIN_READ_LEN = MIN_INSERT_LEN + UMI_LEN + BC_MAX_LEN
print('Using minimum read length: {} (after merge)'.format(MIN_READ_LEN))

Using minimum read length: 31 (after merge)


The minimum alignment score.
The alignment score is calculated as 1 per match, -2 per mismatch, -6 per gap opening and -1 per gap extension.
Better to set relatively low, since additional filtering can be applied later.

In [15]:
MIN_SCORE_ALIGN = 15
print('Using minimum alignemnt score: {}'.format(MIN_SCORE_ALIGN))

Using minimum alignemnt score: 15


### Read some global input

In [16]:
# Read tRNA database information (length, codon etc) into dictionary:
tRNA_data = read_tRNAdb_info(tRNA_database)
# Read index information into dataframe:
index_df = pd.read_excel(index_list_path)
index_df

,type,id,sequence
0,P7_index,D701,ATTACTCG
1,P7_index,D702,TCCGGAGA
2,P7_index,D703,CGCTCATT
3,P7_index,D704,GAGATTCC
4,P7_index,D705,ATTCAGAA
...,...,...,...
226,3primer,C301,GACTGGAGTTCAGACGTGTGCTCTTCCGATCTCTTGTGCAATGTAA...
227,promoter,P01,taatacgactcactatagg
228,5UTR,SD01,tttcatataggaggccgcaa
229,5UTR,SD02,tttcataCATCCCTccgcaa


## Setup - Sample specific if returning to analysis

In [17]:
if os.path.isfile('{}/{}/inp_file_df.xlsx'.format(NBdir,data_dir)) and os.path.isfile('{}/{}/sample_df.xlsx'.format(NBdir,data_dir)):
    # Make a dictionary with paths used for data processing:
    dir_dict = dict(NBdir = NBdir,
                data_dir = data_dir,
                seq_dir = seq_dir,
                AdapterRemoval_dir = AdapterRemoval_dir,
                BC_dir = BC_dir,
                UMI_dir = UMI_dir,
                align_dir = align_dir,
                stats_dir = stats_dir,
                TM_dir = TM_dir,
                charge_dir = charge_dir,
                plotting_dir = plotting_dir)
    # Import the most up-to-date analysis
    inp_file_df= pd.read_excel('{}/{}/inp_file_df.xlsx'.format(NBdir,data_dir))
    sample_df = pd.read_excel('{}/{}/sample_df.xlsx'.format(NBdir,data_dir))
else:
    print('No analysis files generated!')


**NOTE:** Run all the code blocks up to this point and then skip to the analysis block needed.

## Setup - sample specific (Can skip this section if returning to the analysis)
A sample list is defined in an Excel file, in this case named `sample_list.xlsx`.
Here, specify the filename, load it and show the first few rows:

In [18]:
# Enter the name of the sample_list excel file containing all the input data for the experiment.
sample_list_fnam = 'sample_list.xlsx'
sample_df = pd.read_excel('{}/{}'.format(NBdir,sample_list_fnam))
sample_df.head(3)

,sample_name_unique,sample_name,replicate,fastq_mate1_filename,fastq_mate2_filename,P5_index,P7_index,barcode,species,plot_group,hue_name,hue_value,hue_order
0,100p,100p,1,P3_R1.fastq.bz2,P3_R2.fastq.bz2,D501,D703,l2Sp,human,Charge-titration,Percent charge,100p,1
1,85p,85p,1,P3_R1.fastq.bz2,P3_R2.fastq.bz2,D501,D703,l4Sp,human,Charge-titration,Percent charge,85p,2
2,70p,70p,1,P3_R1.fastq.bz2,P3_R2.fastq.bz2,D501,D703,l6Sp,human,Charge-titration,Percent charge,70p,3


### Correct metadata (Optional): 
1. Swap mate1/mate2 files 
2. Reassign barcode
3. Reassign tRNA database: 

<For our case, mate files are correct, barcode is **GCv4**, and tRNA database is **RVHMS04**>

In [19]:
# sample_df['barcode'] = 'l1Sp'
# sample_df['barcode'] = 'l2Sp'
# sample_df['barcode'] = 'l3Sp'
# sample_df['barcode'] = 'l4Sp'
# sample_df['barcode'] = 'l5Sp'
# sample_df['barcode'] = 'l6Sp'
# sample_df['barcode'] = 'l7Sp'
# sample_df['barcode'] = 'l8Sp'
# sample_df['barcode'] = 'l9Sp'
# sample_df['barcode'] = 'l10Sp'
# sample_df['barcode'] = 'l11Sp'
# sample_df['barcode'] = 'l12Sp'
# sample_df['barcode'] = 'GCv3'
#sample_df['barcode'] = 'GCv4'
# sample_df['barcode'] = 'GCv5'
# sample_df['barcode'] = 'ISp1'

# sample_df['species'] = 'PJ35Lib'
#sample_df['species'] = 'RVHMS04Lib'
#sample_df['species'] = 'RVHMS04'

# sample_df.head(3)

To this sample list dataframe add barcode and Illumina index sequences from the index dataframe:

In [20]:
sample_df = index_to_sample_df(sample_df, index_df)

For the MiSeq need to swap mate1 and mate2 columns.

Let's first try without swapping ('if False:'). If we need to swap then change the statement to 'if True:'

In [21]:
if False:
    sample_df['fastq_mate1_filename'], sample_df['fastq_mate2_filename'] = sample_df['fastq_mate2_filename'],sample_df['fastq_mate1_filename']

Extract the filename for the paired-end mates to merge.
In this example there is only one:

In [22]:
# Get filenames from the sample information:
inp_file_df = sample_df[['fastq_mate1_filename', 'fastq_mate2_filename', 'P5_index', 'P7_index', 'P5_index_seq', 'P7_index_seq']].copy().drop_duplicates().reset_index(drop=True)
#Check the sample and input dataframes
inp_file_df.head(3)

,fastq_mate1_filename,fastq_mate2_filename,P5_index,P7_index,P5_index_seq,P7_index_seq
0,P3_R1.fastq.bz2,P3_R2.fastq.bz2,D501,D703,AGGCTATA,CGCTCATT


In [23]:
# Save the files to a readable excel to come back to when rerun sections:

inp_file_df.to_excel('{}/{}/inp_file_df.xlsx'.format(NBdir,data_dir))
sample_df.to_excel('{}/{}/sample_df.xlsx'.format(NBdir,data_dir))

Check to see if all the mate files exist.

In [24]:
for index, row in inp_file_df.iterrows():
    file_path = '{}/{}/{}/{}'.format(NBdir, data_dir, seq_dir, row['fastq_mate1_filename'])
    if os.path.isfile(file_path):
        print('Files exist!')
        continue
    else:
        print('{} doesn\'t exists'.format(row['fastq_mate1_filename']))

Files exist!


Make sure the check if the mate1 and mate2 files are correctly assigned.

In [25]:
sample_df.head(3)

,sample_name_unique,sample_name,replicate,fastq_mate1_filename,fastq_mate2_filename,P5_index,P7_index,barcode,species,plot_group,hue_name,hue_value,hue_order,P5_index_seq,P7_index_seq,barcode_seq
0,100p,100p,1,P3_R1.fastq.bz2,P3_R2.fastq.bz2,D501,D703,l2Sp,human,Charge-titration,Percent charge,100p,1,AGGCTATA,CGCTCATT,GGCTGCCATGCTGTCACG
1,85p,85p,1,P3_R1.fastq.bz2,P3_R2.fastq.bz2,D501,D703,l4Sp,human,Charge-titration,Percent charge,85p,2,AGGCTATA,CGCTCATT,GGCTGCCATGCAAGCTG
2,70p,70p,1,P3_R1.fastq.bz2,P3_R2.fastq.bz2,D501,D703,l6Sp,human,Charge-titration,Percent charge,70p,3,AGGCTATA,CGCTCATT,GGCTGCCATGCTACAG


### Downsampling for quick first-pass processing
This example has 500,000 paired-end reads per file under `data/raw_fastq`.
This is a small amount spread over the multiple samples that have been barcoded and pooled, but it is enough for an example.

In a typical real run each sample has >1e6 reads and a total of more than 100 samples are pooled.
Even simple tasks like uncompressing this amount of reads take a long time.
To quickly check the integrity of the data, whether settings are right etc. it is possible to downsample each datafile using the `downsample_raw_input` function.

There are two ways to perform downsampling at the input stage: using the `downsample_absolute` option or the `downsample_fold` option.
When using the `downsample_absolute` option, samples are taken from the top of the fastq file (for speed reasons) and thus not randomly.
Use the `downsample_fold` option for true random sampling.

It is highly recommended to do a run on downsampled data first e.g. downsampled to 1e4 reads per file.
That way, any error will be detected quickly and when results are satisfactory on the downsampled data the workflow can be performed on the full size data with confidence.

Because this is an example there is no need to downsample the data first; however for reference, below is an example on how to call the `downsample_raw_input` function and downsample to 1e4 reads per file.

In [ ]:
if False:
    sample_df, inp_file_df, seq_dir = downsample_raw_input(sample_df, inp_file_df, NBdir, data_dir, seq_dir_noDS, downsample_absolute=1e4)

### Paths for input/output
Make a dictionary with paths used for data processing:

In [26]:
dir_dict = dict(NBdir = NBdir,
                data_dir = data_dir,
                seq_dir = seq_dir,
                AdapterRemoval_dir = AdapterRemoval_dir,
                BC_dir = BC_dir,
                UMI_dir = UMI_dir,
                align_dir = align_dir,
                stats_dir = stats_dir,
                TM_dir = TM_dir,
                charge_dir = charge_dir,
                plotting_dir = plotting_dir)

## Process tRNA sequencing reads

### Adapter removal and merging
Run AdapterRemoval software to trim off any remaining Illumina adapters (if that has not already been done) and merge the paired-end reads:

In [27]:
# Log the start of the section
log_status("Starting execution: Adaptor Removal")

# Initiate the AdapterRemoval object:
AR_obj = AR_merge(dir_dict, inp_file_df, MIN_READ_LEN, overwrite_dir=True)
# Start AdapterRemoval (here n_jobs is irrelevant because there is only one input file pair):
inp_file_df = AR_obj.run_parallel(n_jobs=1, overwrite=True)

FileNotFoundError: (2, 'No such file or directory')

Statistics about the merging has now been added to the `inp_file_df` dataframe.
Typically, >90 % of all reads are successfully merged.

In [ ]:
inp_file_df

In [ ]:
# Log the end of the section
log_status("Completed execution: Adaptor Removal")

# Save the files to a readable excel to come back to when rerun sections (overwrites the older file):
inp_file_df.to_excel('{}/{}/inp_file_df.xlsx'.format(NBdir,data_dir),index=False)
sample_df.to_excel('{}/{}/sample_df.xlsx'.format(NBdir,data_dir),index=False)

### Sample splitting
Now, each merged file (only one for this example) contains reads from several adapter barcoded samples.
In the following code block the merged file is split into sample specific files.

In [ ]:
# Log the start of the section
log_status("Starting execution: Sample BC Splitting")

# Initiate the barcode splitting object:
BCsplit_obj = BC_split(dir_dict, sample_df, inp_file_df, overwrite_dir=True)
# Split the files (again, n_jobs is irrelevant because there is only one input file):
sample_df, inp_file_df = BCsplit_obj.run_parallel(n_jobs=1)

New columns of statistics have been added to the `inp_file_df` dataframe. Typically, >90 % of all merged reads will be successfully mapped to a barcode and split.<br><br> Time to check if the files are correctly processed. 

In [ ]:
inp_file_df

**NOTE:** This is the point to understand if the mate1 and mate2 files need to be swapped or not. If percent_successfully_merged is high but the percent_BC-mapped is low, it can be either (a) the barcode sequence was incorrect, or (b) the mate 1 and mate 2 files need to be swapped.


Statistics have also also been added to the `sample_df` dataframe.
For example the total number of reads for the sample (N_total) and how many ended on CC vs. CCA.
Since all mature tRNAs end on CCA, and oxidation and 3' cleavage generates 3' CC, the percentage of CCA+CC vs. total reads is a good indidator of sample purity; it is typically >95 %.
The percentage of CCA ending sequences (vs. CCA+CC ending) can be interpreted as the bulk charge

In [ ]:
sample_df

In [ ]:
# Log the end of the section
log_status("Completed execution: Sample BC Splitting")

# Save the files to a readable excel to come back to when rerun sections (overwrites the older file):
inp_file_df.to_excel('{}/{}/inp_file_df.xlsx'.format(NBdir,data_dir),index=False)
sample_df.to_excel('{}/{}/sample_df.xlsx'.format(NBdir,data_dir),index=False)

### Trimming UMI
The reverse transcription was performed with an oligo containing a 10 nt. unique molecular identifier (UMI).
Typically, UMIs are employed to collaped reads from the same molecule when over-sequencing a library made from a very small amount of starting material.
For tRNA sequencing, the amount of starting material should be so high that only few tRNA moleculars are sequenced twice; however, if there were bottlenecks in the library prep that assumption could prove false.
Therefore, the UMI employed for tRNA sequencing is mostly a quality control for the library prep.
It also helps to diversify the start of the read, thereby increasing Illumina sequencing quality, and it randomizes the sequence context for circular ligation, decreasing ligation bias.
Lastly, the UMI is used for count correction later in the stats collection where reads mapping to the same transcript with identical UMI can be count corrected.

But first, we need to trim it off the read before alignment.
Then, we have the option to randomly downsample the resulting trimmed reads to even out the number of reads per sample.
We can either set a maximum number of trimmed reads using the `downsample_absolute` option, or downsample using a percentile of the number of trimmed reads in each sample.
The percentile option is set using `downsample_percentile` and works like the following:  
Observe samples: \[A, B, C, D, E\], with following number of trimmed reads: \[1, 2, 3, 4, 5\]  
Setting `downsample_percentile` to 50, means taking the 50th percentile of \[1, 2, 3, 4, 5\]  
After downsampling, samples \[A, B, C, D, E\] have \[1, 2, 3, 3, 3\] number of trimmed reads.  


In [ ]:
# Log the start of the section
log_status("Starting execution: UMI Trimming")


# Trim 5-prime adapters using a similar method as UMI trimming:

# Make the 5-prime adapters
adapters = [i*'N' + 'AGTCACGATC' for i in range(1, 4)] # GCv4 
# Initiate the 5p trimming object:
_5p_trim_obj = _5p_trim(dir_dict, sample_df, adapters, overwrite_dir=True,downsample_absolute=False)
# Run the 5p trimming in parallel (increase n_jobs to number of processor):
sample_df = _5p_trim_obj.run_parallel(n_jobs=3)


# # Initiate the UMI trimming object:
# UMItrim_obj = UMI_trim(dir_dict, sample_df, overwrite_dir=True, \
#                        downsample_absolute=False)
# # Run the UMI trimming in parallel (increase n_jobs to number of processor):
# sample_df = UMItrim_obj.run_parallel(n_jobs=8)

UMI trimming statistics have been added to the `sample_df` dataframe.
Typically, >95 % of all barcode split reads have a valid UMI sequence i.e. the UMI ends on T or C.

Observe, that the _expected_ number of observed UMIs have been calculated given the length of the UMI and the number reads with valid UMIs.
The percentage of observed vs. expected UMIs is a good validation that the library prep was successful.
It should not be expected to be 100 % because random bases in oligos are [not completely random](https://www.idtdna.com/pages/products/custom-dna-rna/mixed-bases); however, it should typically be >90 %.
Be aware though, that the observed vs. expected UMI percentage depends on the sampling depth and thus the number of reads in a sample will affect this number.
For an example of how the number of reads affect this, see the `utils/code-of-limited-use` folder and check the `UMI_count_analysis` notebook.

In [ ]:
sample_df

In [ ]:
# Log the end of the section
log_status("Completed execution: UMI Trimming")

# Save the files to a readable excel to come back to when rerun sections (overwrites the older file):
inp_file_df.to_excel('{}/{}/inp_file_df.xlsx'.format(NBdir,data_dir), index = False)
sample_df.to_excel('{}/{}/sample_df.xlsx'.format(NBdir,data_dir), index = False)

### Align sequences
Now, the sequences are ready for alignment.
This done using [SWIPE](https://github.com/torognes/swipe/), which is a fast implementation of the Smith-Waterman algorithm.
The gap en gap extension penalties can be adjusted when initiating the alignment object and the match/mismatch score can be adjusted in the score matrix defined in the input above.
However, the current defualt values have been tested and work well so adjustment should not be necessary.

If `common_seqs` is specified, these sequences, when found in each sample, will be counted, skipped and first aligned at the end.
This prevents alignments of common sequences to be run many times and can shorten the alignment time drastically.
When alignment statistics is calculated the common sequences will be added with the their number of observations.

Running this cell should be the slowest computation in this example.
It may take a few minutes to run.

In [ ]:
# Log the start of the section
log_status("Starting execution: Alignment")

# Initiate alignment object:
align_obj = SWIPE_align(dir_dict, tRNA_database, sample_df, \
                        SWIPE_score_mat2, gap_penalty=6, extension_penalty=3, \
                        min_score_align=MIN_SCORE_ALIGN, common_seqs=None, \
                        overwrite_dir=True)
# Start the alignemnt:
sample_df = align_obj.run_parallel(n_jobs=3)

Alignment statistics is added to the `sample_df` dataframe.
Typically, >95% of sequences are mapped and >70% of these have a single annotation.
Those sequences with multiple annotations have exactly the same alignment score when aligned to multiple reference sequences which often happens when there exists multiple transcripts of a tRNAs with the same codon.
The `percent_multiple_codons` quantifies the extend of the more problematic case when the multiple annotations come from transcripts with different codons.
This should only be less than 10% of mapped reads.

In [ ]:
sample_df.head(3)

The statistics is written, per read, as a bz2 compressed .csv file; however, many of the entries in this table will be duplicates and only different by different fastq read ID and UMI sequence.
Therefore, another uncompressed table is made that aggregates all the row that are identical except differing read ID and UMI.

After statistics have been collected for all samples a concatenation of the aggregated data from all samples is made with the name `ALL_stats_aggregate.csv`.
This file is particularly useful because it is relatively compact but contains all information from all samples.

The dataframe returned is the `ALL_stats_aggregate_filtered.csv` which is the aggregated data filtered to contain only the most relevant columnns and requiring that the 3' must be covered and have no 3' non-template bases.

In [ ]:
# Log the end of the section
log_status("Completed execution: Alignment")

# Save the files to a readable excel to come back to when rerun sections (overwrites the older file):

inp_file_df.to_excel('{}/{}/inp_file_df.xlsx'.format(NBdir,data_dir), index = False)
sample_df.to_excel('{}/{}/sample_df.xlsx'.format(NBdir,data_dir), index = False)

### Collect alignment statistics
The alignment results are stored in large JSON files that are practical for parsing but not for data manipulation, plotting etc.
Therefore, it is necessary to collect statistics, add sample information, add UMI info and many more things.
Notice the `common_seqs` argument is also used to fill out the common sequences that were separated out during the alignment phase.

In [ ]:
# Log the start of the section
log_status("Starting execution: Collecting Statistics")

# Initiate the stats collection object:
stats_obj = STATS_collection(dir_dict, tRNA_data, \
                             sample_df, common_seqs=None, \
                             overwrite_dir=True)
# Start collecting stats for each sample:
stats_df = stats_obj.run_parallel(n_jobs=3)

In [ ]:
stats_df.head()

In [ ]:
# Log the end of the section
log_status("Completed execution: Collecting Statistics")

# Save the files to a readable excel to come back to when rerun sections (overwrites the older file):

inp_file_df.to_excel('{}/{}/inp_file_df.xlsx'.format(NBdir,data_dir), index = False)
sample_df.to_excel('{}/{}/sample_df.xlsx'.format(NBdir,data_dir), index = False)
stats_df.to_excel('{}/{}/stats_df.xlsx'.format(NBdir,data_dir),index = False)

### Generating Results
Next, the data can be plotted in order to 1) inspect the data quality and 2) get an overview of the relative tRNA expression and charge.
For this the `TRNA_plot` class was made.
This class allows us to generate some standard plots that are useful for most experiments involving tRNA sequencing.
It utilizes the information in the `sample_df` dataframe to group samples onto the same plot.

The class can the initiated with exclusion filters based on alignment score or alignment gap content and using UMI or read counts.
By using UMI counts instead of read counts, we can correct for PCR duplications; however, typically there is not a big difference between the UMI or read counts.

In [ ]:
# Initiate the plotting object:
plot_obj = TRNA_plot(dir_dict, sample_df, overwrite_dir=True)

In [ ]:
plot_obj.all_stats

#### **Expression & Charge Analysis**

Before moving on, we can dump the charge and relative abundance data to enable custom plotting:

In [ ]:
# Write charge/RPM data grouped by transcript, codon and amino acid:
plot_obj.write_charge_df(df_type='transcript', \
                         fnam='charge-df_tr')
plot_obj.write_charge_df(df_type='codon', \
                         fnam='charge-df_codon')
plot_obj.write_charge_df(df_type='aa', \
                         fnam='charge-df_aa')

#### **Transcript mutation Analysis**
The last element of this analysis is going to be addressing the mismatches between the tRNA sequence of the reference vs. the sequenced read.
Assuming that the reference database is complete and contains no errors the mismatch between reference and read can derive from: 1) biological errors in tRNA synthesis, 2) RT PCR errors, 3) PCR errors, 4) sequencing errors and 5) chemical modification fx depurination.
tRNAs are heavily modified with many modifications disrupting the WatsonÃ¢ÂÂCrick base pairing.
Such modifications will lead to stalling, fall off, misincorporation or skipping in the RT PCR reaction and thus this reaction is by far the most likely process to introduce errors in the read.

Because the read errors are most likely a direct consequence of tRNA modifications, we can use the error type (mismatch/gap/RT stop) and site frequency to measure the level of modification.
Comparison between samples will then allow us to infer differential modifications.
However, the introduction of many mismatches and/or gaps into a read can also make the alignment difficult.
An easy solution is to mask the positions in the reference that are most likely affected such that a mismatch does not lead to alignment penalty.
The alignment run above was actually setup using a masked reference, also indicated by the variable name `tRNA_database_masked`.
This masking was generated a priori using the transcript mutation analysis now presented.

In [ ]:
# Initiate the transcript mutation object:
TM_obj = TM_analysis(dir_dict, sample_df, tRNA_database, pull_default=False, \
                     common_seqs=None, ignore_common_count=False, \
                     overwrite_dir=True)

First, we need to find and count the mutations/gaps/RT stops.
This involves alignment of each read to its respective transcript annotation(s).
Furthermore, we can choose alignment parameters (not shown here) and exclusion criteria, such as only analyzing reads with a unique annotation.

In [ ]:
TM_obj.find_muts(n_jobs=3, unique_anno=False)

After generating transcript mutation data it can be saved.
Saved data can be read at another time, thus avoiding repeating the time consuming analysis.

In [ ]:
# Save transcript mutation data (skip if data already generated):
TM_obj.pickle_muts_write(pickle_name='saved_muts.pickle')

# Read previosly saved transcript mutation data:
TM_obj.pickle_muts_read(pickle_name='saved_muts.pickle')

# Export transcript mutation data to csv 
TM_obj.write_transcript_mut(species='RVHMS04',csv_name='tr-mut_matrix_R1.33', sample_list = ['R133-1','R133-2','R133-3'])
# Export transcript mutation data to csv 
TM_obj.write_transcript_mut(species="RVHMS04",csv_name='tr-mut_matrix_R1.35', sample_list = ['R135-1','R135-2','R135-3'])

### Plotting Results

#### **Abundance**
A common standard plot is a simple view of the relative expression level broken down by either transcript, codon or amino acid identity. This may not be ideal for a library considering close to 4k datapoints.
This plot is generated below:

In [ ]:
# # Grouped plot:
# plot_obj.plot_abundance(plot_name='abundance-plot_codon_grp', group=True, \
#                         plot_type='codon', min_obs=2)
# # Individual sample plot:
# plot_obj.plot_abundance(plot_name='abundance-plot_transcript', group=False, \
#                         plot_type='transcript', min_obs=2, tr_short=False)
# plot_obj.plot_abundance(plot_name='abundance-plot_codon', group=False, \
#                         plot_type='codon', min_obs=2)
# plot_obj.plot_abundance(plot_name='abundance-plot_aa', group=False, \
#                         plot_type='aa', min_obs=2)

#### **Coverage**
Inspired by the coverage plots in Behrens et al. 2021, we could start by plotting the sequence coverage broken down by the amino acid identity of the transcript.
To plot all transcripts on a single plot it is necessary to scale the transcript length such that all are equal.
This scaling is done by mapping all transcripts to the longest transcript.

Below, a Behrens style coverage plot is shown, normalized on the y-axis such that coverage is shown as a percentage of the coverage at the 3' of the transcript.
It shows nicely how the coverage decreases further into the transcript and drops off at particular points which are associated with commonly modified positions.
It also shows the relative abundance of transcripts of an amino acid e.g. higher relative abundance of alanine (A) compared to tryptophan (W).

In [ ]:
plot_obj.plot_coverage(compartment='cyto', plot_type='behrens', y_norm=True, \
                       plot_name='cov-plot_cyto_behrens_norm')
#show_PDF('data/plotting/cov-plot_cyto_behrens_norm.pdf')

Sometimes, it may be desirable to focus on the coverage drop off and show more explicitly the fraction of reads that fully cover a transcript.
For this, we can normalize the coverage such that the difference in relative abundance between different amino acids is equalized.
Furthermore, a different style plot can be generated to enhance the view of coverage drop off.
This plot is referred to as a "needle plot"

Below, such a needle plot is shown.
It clearly shows that the valine transcript (V) has a high read-through while the tryptophan transcript (W) has a very low read-through with two points with large coverage drop off.

In [ ]:
plot_obj.plot_coverage(compartment='cyto', plot_type='needle', aa_norm=True, \
                       plot_name='cov-plot_cyto_needle_aa-norm')
#show_PDF('data/plotting/cov-plot_cyto_needle_aa-norm.pdf')

#### **Aminoacylation/Charge**
Another standard plot is one showing the tRNA charge. This plot might not be good for a library because of the number of data points.
Such a plot, broken down by amino acid and grouped, is shown below:

In [ ]:
# # Grouped plot, per codon:
# plot_obj.plot_abundance(plot_name='charge-plot_codon_grp', group=True, \
#                         plot_type='codon', min_obs=2, charge_plot=True)
# # Grouped plot, per amino acid:
# plot_obj.plot_abundance(plot_name='charge-plot_aa_grp', group=True, \
#                         plot_type='aa', min_obs=2, charge_plot=True)
# #show_PDF('data/plotting/charge-plot_aa_grp.pdf')

#### **Transcript Mutations**

All the data has been generated in the TM_obj can be plotted in many different ways.
One way, would be to plot the positional nucleotide frequency of all reads mapped to a given transcript in the reference database.
Such logo plot is plotted below.
Notice, read gaps appear as a black line.

In [ ]:
TM_obj.plot_transcript_logo(plot_name='tr-mut_logos_R1.33', topN=100, sample_list = ['R133-1','R133-2','R133-3'], species="RVHMS04")
#show_PDF('data/transcript_mutations/tr-mut_logos.pdf')

Alternatively, to view the distribution of base frequency at specific positions with the most significant mutation frequency.

In [ ]:
TM_obj.plot_signif_mut_pos(threshold=0.1, species="RVHMS04", plot_name='tr-signif_nuc_dist_R1.33', png_dpi=False, no_plot_return=False, mito=False, sample_list = ['R133-1','R133-2','R133-3'])

We can also view all the tRNA transcripts in a single plot to see the mutation frequency and coverage across all transcripts. This might not be an easy plot for the library of tRNA sequences.

In [ ]:
obs_mat_df, fig, sorted_anno = TM_obj.plot_transcript_cov(plot_name='tr-cov_matrix_R1.33', species="RVHMS04", sample_list = ['R133-1','R133-2','R133-3'], topN=100)

#### **Comparison/Reproducibility**

Sometimes, it can be useful to plot the relative abundance of two samples in comparison to each other.
For example, for replicates or samples that should have the same relative abundance, the RPM values should fall on a straight line.
This is plotted below, with each transcript as a datapoint, a linear regression line in blue and a one-to-one correspondance as a red line:

In [ ]:
plot_obj.plot_abundance_corr(sample_unique_pairs=[['ssR133-T1','ssR133-T1','ssR133-T2'],
                                                  ['ssR133-T2','ssR133-T3','ssR133-T3']], \
                             plot_type='transcript', charge_plot=False, log=True, \
                             plot_name='abundance-corr-plot_ssR133', \
                             min_obs=10)

A similar plot can be made showing charge instead of RPM:

In [ ]:
# This plot does not make sense using simulated data.
#plot_obj.plot_abundance_corr(sample_unique_pairs=[['R133-1'], ['R133-2']], \
#                             plot_type='aa', charge_plot=True, log=False, \
#                             plot_name='charge-corr-plot_aa', \
#                             min_obs=500, one2one_corr=True)
#show_PDF('data/plotting/charge-corr-plot_aa.pdf')

Finally, we can look into some of the features of the library prep.
First, let us plot a logo over the positional nucleotide use of the UMI:

In [ ]:
# This plot does not make sense using simulated data.
# plot_obj.plot_UMI_logo(plot_name='UMI_logo')
# show_PDF('data/plotting/UMI_logo.pdf')

The UMI logo plot shows approxiamate random nucleotide use on all positions except the 3' nt. which is always a random pyrimidine.
This is also what should be expected given the RT oligo design.

Next, we can plot the 5' non-template nucleotides.
These are added to the end of the cDNA by the reverse transcriptase during the RT-PCR reaction.
Here, we only include up to the 99th percentile of the longest 5' non-template nucleotide additions:

In [ ]:
# plot_obj.plot_non_temp(end='5p', plot_name='_5p-non-template_logo', \
#                        seq_len_percentile=99)
# #show_PDF('data/plotting/_5p-non-template_logo.pdf')

We observe that the most commonly observed 5' non-template nucleotide is T, which means that the reverse transcriptase adds A to the 3' of the cDNA.
This is similar to the observation made by Behrens et al. 2021.

Next, we can look at the 3' non-template nucleotides.
These are not added by the reverse transcriptase but are likely either erronous CCA additions or additional bases in the adapter.
Plotted below and only showing sequences with 3' coverage:

In [ ]:
# plot_obj.plot_non_temp(end='3p', plot_name='_3p-non-template_logo', \
#                        seq_len_percentile=99.9, _3p_cover=True)
# #show_PDF('data/plotting/_3p-non-template_logo.pdf')

From this, we can see that 3' non-template nucleotides are a rare occurence with mostly one or two nucleotides added.
Had there been many more sequences with 3' non-template nucleotides it would have indicated a problem, possibly with the quality of the adapter oligo.